# F03 SIGISMUND — Réacteur Remotion SVG Neon
## PENTERACT DORN V3 — VIIe Légion

**Rôle** : Rendu Remotion 60fps, courbes SVG neon, timing dynamique, caméra virtuelle.

**Entrées** : `F03_SIGISMUND/IN/plan_de_vol.json` (validé par F02) + `IN/*.png`  
**Sorties** : `F03_SIGISMUND/OUT/video_render.mp4`

---
### Avant de lancer
1. Monte ton Google Drive (cellule 1)
2. Vérifie que `F03_SIGISMUND/IN/plan_de_vol.json` existe avec `validated_by_magos: true`
3. Dépose tes PNG dans `F03_SIGISMUND/IN/` si tu utilises des assets
4. Choix du mode : `--mode direct` (1 worker, rapide) ou `--mode modal` (3 workers)
5. Lance toutes les cellules dans l'ordre

In [ ]:
# CELLULE 1 — Montage Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive monté.')

In [ ]:
# CELLULE 2 — Configuration
DRIVE_BASE = '/content/drive/MyDrive/DRIVE_DORN'  # ← adapter si besoin
MODE       = 'direct'  # 'direct' ou 'modal'
print(f'Drive base : {DRIVE_BASE}')
print(f'Mode rendu : {MODE}')

In [ ]:
# CELLULE 3 — Installation Node.js + npm (si absent)
import subprocess, sys
result = subprocess.run(['node', '--version'], capture_output=True)
if result.returncode != 0:
    print('Installation Node.js 20.x ...')
    subprocess.run(['bash', '-c',
        'curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y nodejs'],
        check=True)
else:
    print(f'Node.js déjà présent : {result.stdout.decode().strip()}')

# Vérifier npm
r2 = subprocess.run(['npm', '--version'], capture_output=True)
print(f'npm : {r2.stdout.decode().strip()}')

In [ ]:
# CELLULE 4 — Copie du script depuis Drive + npm install
import shutil, os
from pathlib import Path

codebase_src = Path(DRIVE_BASE) / 'F03_SIGISMUND' / 'CODEBASE'
codebase_dst = Path('/content/F03_CODEBASE')

# Copier tout le codebase Remotion vers /content
if codebase_dst.exists():
    shutil.rmtree(codebase_dst)
shutil.copytree(codebase_src, codebase_dst)
print(f'Codebase copié : {codebase_dst}')

# Copier le script principal
script_dst = '/content/drn_f03_sigismund.py'
shutil.copy2(codebase_dst / 'drn_f03_sigismund.py', script_dst)
print(f'Script copié : {script_dst}')

# npm install (Remotion + mathjs)
print('npm install ...')
subprocess.run(['npm', 'install', '--prefer-offline'], cwd=str(codebase_dst), check=True)
print('npm install OK')

In [ ]:
# CELLULE 5 — Lancement du rendu F03 SIGISMUND
import subprocess, sys

result = subprocess.run(
    [sys.executable, script_dst,
     '--mode',       MODE,
     '--drive-base', DRIVE_BASE],
    capture_output=False
)

if result.returncode == 0:
    print('\n✓ F03 SIGISMUND — RENDU OK')
    print(f'→ Vidéo dans : {DRIVE_BASE}/F03_SIGISMUND/OUT/video_render.mp4')
    print('→ Étape suivante : F04A INWIT (viewer + sélecteur vitesse)')
else:
    print('\n✗ F03 SIGISMUND — RENDU FAIL — corriger les erreurs ci-dessus')

In [ ]:
# CELLULE 6 — Transit CRS_CUSTOS (check-in F03)
# À lancer UNIQUEMENT si la cellule 5 s'est terminée avec RENDU OK
import shutil, subprocess, sys, os
from pathlib import Path

custos_src = Path(DRIVE_BASE) / 'CRS_CUSTOS.py'
shutil.copy2(custos_src, '/content/CRS_CUSTOS.py')

result = subprocess.run(
    [sys.executable, '/content/CRS_CUSTOS.py',
     '--frigate', 'F03', '--mode', 'check-in', '--drive-base', DRIVE_BASE],
    capture_output=False
)
if result.returncode == 0:
    print('\n✓ CRS_CUSTOS — F03 check-in OK — Transit autorisé vers F04')
else:
    print('\n✗ CRS_CUSTOS — F03 check-in FAIL')